In [9]:
import torch
import torch.nn as nn

from mslandcover.models import DeepLabV3Plus, ResNetBackbone, ProjectionHead
from mslandcover.utils import load_pth

In [79]:
weights_path = './weights/resnet152_202505/hires_simclr_bands3_size256_batch128_randinitfalse/resnet152/hires_simclr.pth'

backbone = ResNetBackbone(in_channels=3)
weights = torch.load(weights_path, map_location='cpu')

# model = DeepLabV3Plus(
#     backbone=ResNetBackbone(
#         arch='resnet152',
#         pretrained=False,
#         out_channels=2048,
#         norm_layer=nn.BatchNorm2d,
#         dilation=True
#     ),
#     num_classes=17,
#     aux_loss=True,
#     norm_layer=nn.BatchNorm2d
# )

/var/folders/1j/_j01624x6y33h54v02qwk81r0000gn/T/ipykernel_22844/3289691822.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(weights_path, map_locati

In [80]:
def adjust_backbone_weights(weights):
    new_weights = {}
    for key in weights.keys():
        if key.startswith('encoder.'):
            new_key = key.replace('encoder.', '')
            
            # initial layers should match up
            if new_key == 'conv1.weight':
                new_key = 'initial.0.weight'
            elif new_key == 'bn1.weight':
                new_key = 'initial.1.weight'
            elif new_key == 'bn1.bias':
                new_key = 'initial.1.bias'
            elif new_key == 'bn1.running_mean':
                new_key = 'initial.1.running_mean'
            elif new_key == 'bn1.running_var':
                new_key = 'initial.1.running_var'
            elif new_key == 'bn1.num_batches_tracked':
                new_key = 'initial.1.num_batches_tracked'
                
            new_weights[new_key] = weights[key]
        
    return new_weights

In [81]:
backbone.load_state_dict(adjust_backbone_weights(weights))

<All keys matched successfully>

In [82]:
model = DeepLabV3Plus(
    backbone=backbone,
    num_classes=8,
)

In [38]:
x = torch.randn(2, 4, 256, 256)
model.eval()
with torch.no_grad():
    y = model(x)

In [40]:
y.shape

torch.Size([2, 8, 256, 256])

In [83]:
import pandas as pd
splits_df = pd.read_csv('./data/splits/splits.csv')

In [86]:
splits_df = splits_df.loc[splits_df['n_train'] == 250]

In [88]:
splits_df.loc[splits_df['fold'] == 1]

,n_train,fold,split_1,split_2,split_3,split_4
0,250,1,train,val,val,val


In [91]:
# select column where entry is 'train'
train_splits = [split for split in splits_df.columns if splits_df[split].iloc[0] == 'train']
val_splits = [split for split in splits_df.columns if splits_df[split].iloc[0] == 'val']
print("Train splits:", train_splits)
print("Validation splits:", val_splits)

Train splits: ['split_1']
Validation splits: ['split_2', 'split_3', 'split_4']


In [ ]:
from glob import glob
input_paths = [file for split in train_splits for file in glob(f'./data/splits/{split}/input/*.tif')]
target_paths = [file for split in train_splits for file in glob(f'./data/splits/{split}/target/*.tif')]

In [102]:
print(len(input_paths))

250


In [ ]:
splits_df.loc[splits_df['spl

0      True
1     False
2     False
3     False
4      True
5     False
6     False
7     False
8     False
9     False
10     True
11    False
12    False
13    False
Name: fold, dtype: bool